# Autograd — Automatic Differentiation
PyTorch tracks operations on tensors with `requires_grad=True` and computes gradients automatically via `.backward()`.

In [ ]:
import torch

## 1. requires_grad and grad_fn

In [ ]:
x = torch.tensor(3.0, requires_grad=True)
print(x)              # tensor(3., requires_grad=True)

y = x ** 2            # y = x²
print(y)              # tensor(9., grad_fn=<PowBackward0>)
print(y.grad_fn)      # PowBackward0 — PyTorch remembers how y was built

## 2. .backward() — compute gradients

In [ ]:
x = torch.tensor(3.0, requires_grad=True)
y = x ** 2

y.backward()          # dy/dx = 2x
print(x.grad)         # tensor(6.) — gradient stored in .grad

In [ ]:
# Chain rule — multi-step
x = torch.tensor(2.0, requires_grad=True)
a = x ** 3          # da/dx = 3x²
b = a + 5           # db/da = 1
c = b * 2           # dc/db = 2

c.backward()
# dc/dx = dc/db * db/da * da/dx = 2 * 1 * 3*(2²) = 24
print(x.grad)       # tensor(24.)

## 3. Gradient Accumulation — zero_grad()

In [ ]:
x = torch.tensor(2.0, requires_grad=True)

for _ in range(3):
    y = x ** 2
    y.backward()
    print(x.grad)   # 4, 8, 12 — grads accumulate!

# Fix: zero out before each backward
x = torch.tensor(2.0, requires_grad=True)
for _ in range(3):
    y = x ** 2
    y.backward()
    print(x.grad)   # 4, 4, 4
    x.grad.zero_()  # in-place zero

## 4. torch.no_grad() — disable tracking

In [ ]:
x = torch.tensor(3.0, requires_grad=True)

with torch.no_grad():
    y = x ** 2
    print(y.requires_grad)  # False — no grad tracking inside block

# Use during inference and evaluation — saves memory and compute

## 5. Vector Gradients

In [ ]:
x = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
y = x ** 2          # element-wise
z = y.sum()         # scalar required for .backward()

z.backward()
print(x.grad)       # tensor([2., 4., 6.]) — dz/dx_i = 2*x_i

## 6. Manual Gradient Descent
Shows what PyTorch optimizers do under the hood.

In [ ]:
# Learn w such that w * 2 ≈ 10  (i.e., w ≈ 5)
w = torch.tensor(0.0, requires_grad=True)
lr = 0.1

for step in range(20):
    pred = w * 2
    loss = (pred - 10) ** 2     # MSE-like

    loss.backward()

    with torch.no_grad():
        w -= lr * w.grad        # gradient descent step
        w.grad.zero_()

    if step % 5 == 0:
        print(f"step {step:2d}  w={w.item():.4f}  loss={loss.item():.4f}")

print(f"\nFinal w: {w.item():.4f}  (expected 5.0)")

## 7. .detach() — break gradient flow

In [ ]:
x = torch.tensor(3.0, requires_grad=True)
y = x ** 2

y_detached = y.detach()  # new tensor, same data, no grad
print(y_detached.requires_grad)  # False

# Common use: extract tensor value for logging/plotting without side effects